# 02 · Experiment
Trains all models and writes their predictions to `results/`. The saved results are included in the repository; `03_results.ipynb` reproduces the article from them without running this notebook. A GPU is recommended for the LSTM.

| Task | Issued | Target |
|---|---|---|
| intraday | every hour | next 1–6 hours |
| dayahead | daily at 10:00 UTC | 24 hours of the next day |

| Configuration | Inputs |
|---|---|
| no_nwp | measurements up to the issue time + solar geometry |
| nwp | + MEPS forecast from the 00 UTC run available at the issue time |

| Fold | Training | Test |
|---|---|---|
| 1 | 2017–2018 | 2019 |
| 2 | 2017–2019 | 2020 |
| 3 | 2017–2020 | 2021 |

Hyperparameters are tuned in each fold on the last training year (nested); the test year is never seen. Finished tuning runs are cached in `results/tuning/`.

## Setup

In [ ]:
import os, sys

REPO = "helsinki-multihorizon-pv"
REPO_URL = f"https://github.com/rhmano/{REPO}.git"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO):
        os.system(f"git clone -q {REPO_URL}")
    ROOT = os.path.abspath(REPO)
    os.system(f"pip install -q xgboost")
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "src"))

from multistep_core import *
print("NWP shift:", CFG.nwp_shift_hours, "h | results folder:", CFG.results_dir)

## 1. Hourly series and sample tables

In [ ]:
BASE = prepare_base(CFG)
NWP = load_nwp_table(CFG)
if NWP is None:
    log.info(t("no_nwp_table"))
SAMPLES = {task: build_samples(BASE, task, CFG, NWP) for task in TASKS}
CONFIGS = ("no_nwp", "nwp") if NWP is not None else ("no_nwp",)

## 2. Reference choice (2017 → 2018, test years unseen)

In [ ]:
REF = {task: choose_reference(task, SAMPLES[task], CFG) for task in TASKS}

## 3. References
Smart persistence and climatology, built in each fold from the training years only.

In [ ]:
for task in TASKS:
    for config in CONFIGS:
        run_references(task, config, SAMPLES[task], CFG)

## 4. MLR, KNN, RF and XGBoost
The grids are in `param_grids` (`multistep_core.py`). A warning is printed if a selected value lies at the edge of its grid.

In [ ]:
for algo in ("MLR", "KNN", "RandomForest", "XGBoost"):
    for task in TASKS:
        for config in CONFIGS:
            run_algorithm(algo, task, config, BASE, SAMPLES[task], CFG)

## 5. SVR
Training (including tuning) uses a random subsample of at most `svr_max_train` = 15,000 rows.

In [ ]:
for task in TASKS:
    for config in CONFIGS:
        run_algorithm("SVR", task, config, BASE, SAMPLES[task], CFG)

## 6. LSTM (GPU)
One model predicts all horizons jointly (MIMO). Units and look-back are tuned with seed 42; the selected structure is trained with five seeds.

In [ ]:
for task in TASKS:
    for config in CONFIGS:
        run_algorithm("LSTM", task, config, BASE, SAMPLES[task], CFG)

## 7. Learning curve (test 2021, nwp, 1–4 training years)

In [ ]:
LC = learning_curve(BASE, SAMPLES, CFG)

## 8. Integrity check
All results and prediction files must exist before running `03_results.ipynb`.

In [ ]:
import glob

missing = []
for task in TASKS:
    for config in ("no_nwp", "nwp"):
        for algo in ALGORITHMS:
            f = os.path.join(CFG.results_dir, f"results_{task}_{config}_{algo}.csv")
            p = os.path.join(CFG.predictions_dir, f"{task}_{config}_{algo}.csv.gz")
            if not (os.path.exists(f) and os.path.exists(p)):
                missing.append(f"{task} | {config} | {algo}")
print("Missing:", missing if missing else "none")
print("Tuning files:", len(glob.glob(os.path.join(CFG.results_dir, "tuning", "*.json"))))